# Lesson 1: Router Engine

Build a simple agentic RAG router that chooses between **summarization** and **vector Q&A** over a single PDF, using **Gemini** via LlamaIndex.

## Setup

Uses `GOOGLE_API_KEY` from your project `.env` (same pattern as week_1 / week_2).

Install deps once from this folder:

```bash
pip install -r requirements.txt
```

Place `metagpt.pdf` in this `week_3/` directory (download tip below).

In [16]:
from helper import get_google_api_key, configure_settings

GOOGLE_API_KEY = get_google_api_key()
llm, embed_model = configure_settings()
print(f"LLM: Gemini | Embeddings (local): {embed_model.model_name}")

2026-07-14 10:07:01,837 - INFO - HTTP Request: GET https://generativelanguage.googleapis.com/v1beta/models/gemma-4-31b-it "HTTP/1.1 200 OK"
2026-07-14 10:07:02,393 - INFO - HTTP Request: HEAD https://huggingface.co/BAAI/bge-small-en-v1.5/resolve/main/modules.json "HTTP/1.1 307 Temporary Redirect"
2026-07-14 10:07:02,433 - INFO - HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/BAAI/bge-small-en-v1.5/5c38ec7c405ec4b44b94cc5a9bb96e735b38267a/modules.json "HTTP/1.1 200 OK"
2026-07-14 10:07:02,697 - INFO - HTTP Request: HEAD https://huggingface.co/BAAI/bge-small-en-v1.5/resolve/main/config_sentence_transformers.json "HTTP/1.1 307 Temporary Redirect"
2026-07-14 10:07:02,740 - INFO - HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/BAAI/bge-small-en-v1.5/5c38ec7c405ec4b44b94cc5a9bb96e735b38267a/config_sentence_transformers.json "HTTP/1.1 200 OK"
2026-07-14 10:07:02,743 - INFO - Loading SentenceTransformer model from BAAI/bge-small-en-v1.5.
2026-07-14 10:07

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

2026-07-14 10:07:04,994 - INFO - HTTP Request: HEAD https://huggingface.co/BAAI/bge-small-en-v1.5/resolve/main/processor_config.json "HTTP/1.1 404 Not Found"
2026-07-14 10:07:05,264 - INFO - HTTP Request: HEAD https://huggingface.co/BAAI/bge-small-en-v1.5/resolve/main/preprocessor_config.json "HTTP/1.1 404 Not Found"
2026-07-14 10:07:05,535 - INFO - HTTP Request: HEAD https://huggingface.co/BAAI/bge-small-en-v1.5/resolve/main/video_preprocessor_config.json "HTTP/1.1 404 Not Found"
2026-07-14 10:07:05,855 - INFO - HTTP Request: HEAD https://huggingface.co/BAAI/bge-small-en-v1.5/resolve/main/preprocessor_config.json "HTTP/1.1 404 Not Found"
2026-07-14 10:07:06,163 - INFO - HTTP Request: HEAD https://huggingface.co/BAAI/bge-small-en-v1.5/resolve/main/tokenizer_config.json "HTTP/1.1 307 Temporary Redirect"
2026-07-14 10:07:06,206 - INFO - HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/BAAI/bge-small-en-v1.5/5c38ec7c405ec4b44b94cc5a9bb96e735b38267a/tokenizer_config.json 

LLM: Gemini | Embeddings (local): BAAI/bge-small-en-v1.5


In [17]:
import nest_asyncio

nest_asyncio.apply()

## Load Data

To download the MetaGPT paper:

```bash
# from the week_3 directory
curl -L "https://openreview.net/pdf?id=VtmBAGCN7o" -o metagpt.pdf
```

Or use the cell below.

In [18]:
# !curl -L "https://openreview.net/pdf?id=VtmBAGCN7o" -o metagpt.pdf

from helper import load_pdf

# Important: use PDFReader (via load_pdf), not raw SimpleDirectoryReader —
# otherwise the PDF binary is treated as text and explodes into thousands of chunks.
documents = load_pdf("metagpt.pdf")
print(f"Loaded {len(documents)} pages")


Loaded 29 pages


## Define LLM and Embedding model

`configure_settings()` sets **Gemini for the LLM** and a **local HuggingFace embedding model** (no embedding API quota). Below we chunk the document.

In [19]:
from llama_index.core.node_parser import SentenceSplitter

splitter = SentenceSplitter(chunk_size=1024)
nodes = splitter.get_nodes_from_documents(documents)
print(f"Created {len(nodes)} nodes")

Created 34 nodes


In [20]:
from llama_index.core import Settings
from llama_index.llms.google_genai import GoogleGenAI
from helper import DEFAULT_LLM_MODEL, DEFAULT_EMBED_MODEL, get_embed_model

# Explicit (same as configure_settings) — useful when teaching
Settings.llm = GoogleGenAI(model=DEFAULT_LLM_MODEL, api_key=GOOGLE_API_KEY)
Settings.embed_model = get_embed_model(DEFAULT_EMBED_MODEL)  # local, free


2026-07-14 10:12:10,765 - INFO - HTTP Request: GET https://generativelanguage.googleapis.com/v1beta/models/gemma-4-31b-it "HTTP/1.1 200 OK"
2026-07-14 10:12:11,109 - INFO - HTTP Request: HEAD https://huggingface.co/BAAI/bge-small-en-v1.5/resolve/main/modules.json "HTTP/1.1 307 Temporary Redirect"
2026-07-14 10:12:11,148 - INFO - HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/BAAI/bge-small-en-v1.5/5c38ec7c405ec4b44b94cc5a9bb96e735b38267a/modules.json "HTTP/1.1 200 OK"
2026-07-14 10:12:11,456 - INFO - HTTP Request: HEAD https://huggingface.co/BAAI/bge-small-en-v1.5/resolve/main/config_sentence_transformers.json "HTTP/1.1 307 Temporary Redirect"
2026-07-14 10:12:11,495 - INFO - HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/BAAI/bge-small-en-v1.5/5c38ec7c405ec4b44b94cc5a9bb96e735b38267a/config_sentence_transformers.json "HTTP/1.1 200 OK"
2026-07-14 10:12:11,499 - INFO - Loading SentenceTransformer model from BAAI/bge-small-en-v1.5.
2026-07-14 10:12

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

2026-07-14 10:12:13,689 - INFO - HTTP Request: HEAD https://huggingface.co/BAAI/bge-small-en-v1.5/resolve/main/processor_config.json "HTTP/1.1 404 Not Found"
2026-07-14 10:12:13,964 - INFO - HTTP Request: HEAD https://huggingface.co/BAAI/bge-small-en-v1.5/resolve/main/preprocessor_config.json "HTTP/1.1 404 Not Found"
2026-07-14 10:12:14,231 - INFO - HTTP Request: HEAD https://huggingface.co/BAAI/bge-small-en-v1.5/resolve/main/video_preprocessor_config.json "HTTP/1.1 404 Not Found"
2026-07-14 10:12:14,503 - INFO - HTTP Request: HEAD https://huggingface.co/BAAI/bge-small-en-v1.5/resolve/main/preprocessor_config.json "HTTP/1.1 404 Not Found"
2026-07-14 10:12:14,774 - INFO - HTTP Request: HEAD https://huggingface.co/BAAI/bge-small-en-v1.5/resolve/main/tokenizer_config.json "HTTP/1.1 307 Temporary Redirect"
2026-07-14 10:12:14,814 - INFO - HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/BAAI/bge-small-en-v1.5/5c38ec7c405ec4b44b94cc5a9bb96e735b38267a/tokenizer_config.json 

## Define Summary Index and Vector Index over the Same Data

In [21]:
from llama_index.core import SummaryIndex, VectorStoreIndex

summary_index = SummaryIndex(nodes)
vector_index = VectorStoreIndex(nodes)

## Define Query Engines and Set Metadata

In [22]:
summary_query_engine = summary_index.as_query_engine(
    response_mode="tree_summarize",
    use_async=False,  # sequential calls play nicer with low RPM quotas
)
vector_query_engine = vector_index.as_query_engine()


In [23]:
from llama_index.core.tools import QueryEngineTool

summary_tool = QueryEngineTool.from_defaults(
    query_engine=summary_query_engine,
    description="Useful for summarization questions related to MetaGPT",
)

vector_tool = QueryEngineTool.from_defaults(
    query_engine=vector_query_engine,
    description="Useful for retrieving specific context from the MetaGPT paper.",
)

## Define Router Query Engine

In [28]:
from llama_index.core.query_engine.router_query_engine import RouterQueryEngine
from llama_index.core.selectors import LLMSingleSelector

query_engine = RouterQueryEngine(
    selector=LLMSingleSelector.from_defaults(),
    query_engine_tools=[
        summary_tool,
        vector_tool,
    ],
    verbose=True,
)

In [29]:
response = query_engine.query("What is the summary of the document?")
print(str(response))

2026-07-14 10:28:45,006 - INFO - AFC is enabled with max remote calls: 10.
2026-07-14 10:28:53,452 - INFO - HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemma-4-31b-it:generateContent "HTTP/1.1 200 OK"
2026-07-14 10:28:53,456 - INFO - Selecting query engine 0: The question asks for a summary of the document, and choice (1) explicitly states it is useful for summarization questions..
2026-07-14 10:28:53,488 - INFO - AFC is enabled with max remote calls: 10.


Selecting query engine 0: The question asks for a summary of the document, and choice (1) explicitly states it is useful for summarization questions..


2026-07-14 10:29:20,091 - INFO - HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemma-4-31b-it:generateContent "HTTP/1.1 200 OK"


MetaGPT is a meta-programming framework for multi-agent collaboration based on Large Language Models (LLMs) that incorporates human-like Standardized Operating Procedures (SOPs) to streamline workflows and reduce logic inconsistencies and hallucinations. The framework utilizes an assembly line paradigm, assigning specialized roles—such as Product Manager, Architect, Project Manager, Engineer, and QA Engineer—to break down complex tasks into actionable subtasks.

Key features of MetaGPT include:
*   **Structured Communication:** Rather than relying on unconstrained natural language dialogue, agents generate structured outputs like Product Requirements Documents (PRDs), design artifacts, and interface specifications.
*   **Communication Protocol:** To enhance efficiency and avoid information overload, MetaGPT uses a shared message pool and a publish-subscribe mechanism, allowing agents to access relevant information based on their role profiles.
*   **Executable Feedback:** The framework

In [30]:
print(len(response.source_nodes))

34


In [31]:
response = query_engine.query(
    "How do agents share information with other agents?"
)
print(str(response))

2026-07-14 10:29:42,323 - INFO - AFC is enabled with max remote calls: 10.
2026-07-14 10:29:50,325 - INFO - HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemma-4-31b-it:generateContent "HTTP/1.1 200 OK"
2026-07-14 10:29:50,329 - INFO - Selecting query engine 1: The question asks for a specific mechanism ('How do agents share information'), which requires retrieving specific context from the MetaGPT paper rather than a general summary..
2026-07-14 10:29:50,410 - INFO - AFC is enabled with max remote calls: 10.


Selecting query engine 1: The question asks for a specific mechanism ('How do agents share information'), which requires retrieving specific context from the MetaGPT paper rather than a general summary..


2026-07-14 10:30:03,230 - INFO - HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemma-4-31b-it:generateContent "HTTP/1.1 200 OK"


Agents share information through structured communication using documents and diagrams rather than dialogue. They utilize a shared message pool to exchange messages directly, where agents publish their structured messages and transparently access messages from other entities. To prevent information overload, a subscription mechanism allows agents to use their role profiles and role-specific interests to select and extract only the relevant information they need to follow.


## Let's put everything together

`utils.get_router_query_engine` wraps the steps above and uses Gemini by default.

In [32]:
from utils import get_router_query_engine

query_engine = get_router_query_engine("metagpt.pdf")

2026-07-14 10:30:54,499 - INFO - HTTP Request: GET https://generativelanguage.googleapis.com/v1beta/models/gemma-4-31b-it "HTTP/1.1 200 OK"
2026-07-14 10:30:54,906 - INFO - HTTP Request: HEAD https://huggingface.co/BAAI/bge-small-en-v1.5/resolve/main/modules.json "HTTP/1.1 307 Temporary Redirect"
2026-07-14 10:30:54,949 - INFO - HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/BAAI/bge-small-en-v1.5/5c38ec7c405ec4b44b94cc5a9bb96e735b38267a/modules.json "HTTP/1.1 200 OK"
2026-07-14 10:30:55,222 - INFO - HTTP Request: HEAD https://huggingface.co/BAAI/bge-small-en-v1.5/resolve/main/config_sentence_transformers.json "HTTP/1.1 307 Temporary Redirect"
2026-07-14 10:30:55,265 - INFO - HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/BAAI/bge-small-en-v1.5/5c38ec7c405ec4b44b94cc5a9bb96e735b38267a/config_sentence_transformers.json "HTTP/1.1 200 OK"
2026-07-14 10:30:55,270 - INFO - Loading SentenceTransformer model from BAAI/bge-small-en-v1.5.
2026-07-14 10:30

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

2026-07-14 10:30:57,541 - INFO - HTTP Request: HEAD https://huggingface.co/BAAI/bge-small-en-v1.5/resolve/main/processor_config.json "HTTP/1.1 404 Not Found"
2026-07-14 10:30:57,811 - INFO - HTTP Request: HEAD https://huggingface.co/BAAI/bge-small-en-v1.5/resolve/main/preprocessor_config.json "HTTP/1.1 404 Not Found"
2026-07-14 10:30:58,090 - INFO - HTTP Request: HEAD https://huggingface.co/BAAI/bge-small-en-v1.5/resolve/main/video_preprocessor_config.json "HTTP/1.1 404 Not Found"
2026-07-14 10:30:58,354 - INFO - HTTP Request: HEAD https://huggingface.co/BAAI/bge-small-en-v1.5/resolve/main/preprocessor_config.json "HTTP/1.1 404 Not Found"
2026-07-14 10:30:58,684 - INFO - HTTP Request: HEAD https://huggingface.co/BAAI/bge-small-en-v1.5/resolve/main/tokenizer_config.json "HTTP/1.1 307 Temporary Redirect"
2026-07-14 10:30:58,727 - INFO - HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/BAAI/bge-small-en-v1.5/5c38ec7c405ec4b44b94cc5a9bb96e735b38267a/tokenizer_config.json 

In [33]:
response = query_engine.query("Tell me about the ablation study results?")
print(str(response))

2026-07-14 10:31:20,028 - INFO - AFC is enabled with max remote calls: 10.
2026-07-14 10:31:27,466 - INFO - HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemma-4-31b-it:generateContent "HTTP/1.1 200 OK"
2026-07-14 10:31:27,471 - INFO - Selecting query engine 1: The question asks for 'ablation study results', which is a request for specific context and detailed findings typically found within a research paper, making choice (2) the most relevant..
2026-07-14 10:31:27,521 - INFO - AFC is enabled with max remote calls: 10.


Selecting query engine 1: The question asks for 'ablation study results', which is a request for specific context and detailed findings typically found within a research paper, making choice (2) the most relevant..


2026-07-14 10:31:57,197 - INFO - HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemma-4-31b-it:generateContent "HTTP/1.1 200 OK"


The provided text details experiments regarding the impact of instruction levels and the use of different LLM backends:

**Impact of Instruction Levels**
Experiments comparing high-level prompts (e.g., "Create a brick breaker game") and detailed prompts (which include specific game mechanics) showed that detailed prompts lead to better software projects. This is evidenced by:
*   **Executability:** Detailed prompts achieved a score of 4.0 compared to 3.8 for high-level prompts.
*   **Productivity:** Detailed prompts resulted in a lower productivity ratio (118.0 vs. 163.8), which is considered better.
*   **Observation:** While detailed prompts provide clearer requirements and functions, simple high-level inputs still generate software with a comparable executability rating.

**LLM Agent Backends**
When testing MetaGPT on SoftwareDev with different LLMs, the results were as follows:
*   **GPT-4:** Achieved the highest executability (3.8) and the lowest number of revisions (1.2).
*   **G

In [34]:
response = query_engine.query("Can you give summary of section 2 (Related work)")
print(str(response))

2026-07-14 10:32:34,611 - INFO - AFC is enabled with max remote calls: 10.
2026-07-14 10:32:42,823 - INFO - HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemma-4-31b-it:generateContent "HTTP/1.1 200 OK"
2026-07-14 10:32:42,827 - INFO - Selecting query engine 0: The question explicitly asks for a 'summary' of a specific section, which directly aligns with the purpose of choice (1), which is for summarization questions related to MetaGPT..
2026-07-14 10:32:42,855 - INFO - AFC is enabled with max remote calls: 10.


Selecting query engine 0: The question explicitly asks for a 'summary' of a specific section, which directly aligns with the purpose of choice (1), which is for summarization questions related to MetaGPT..


2026-07-14 10:33:12,618 - INFO - HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemma-4-31b-it:generateContent "HTTP/1.1 200 OK"


Section 2 discusses related work in two primary areas: automatic programming and LLM-based multi-agent frameworks.

**Automatic Programming**
Automatic programming has evolved from early systems like "PROW" in 1969 to modern approaches utilizing natural language processing and industry tools such as Microsoft Copilot. Recent developments include LLM-based agents that use chain-of-thought prompts to create action plans and reasoning trajectories, such as ReAct and Reflexion, as well as ToolFormer, which learns to use external tools via APIs. While some existing role-play frameworks for programming have improved productivity, they often lack structured output formats, which can make it difficult to resolve complex software engineering issues.

**LLM-Based Multi-Agent Frameworks**
There is significant interest in using multi-agent discussions to improve the problem-solving abilities of LLMs. Various frameworks have explored different applications, such as:
*   **Social and Behavioral Simu

In [35]:
response = query_engine.query("Which related work contribute most to the published paper?")
print(str(response))

2026-07-14 10:33:31,830 - INFO - AFC is enabled with max remote calls: 10.
2026-07-14 10:33:39,361 - INFO - HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemma-4-31b-it:generateContent "HTTP/1.1 200 OK"
2026-07-14 10:33:39,365 - INFO - Selecting query engine 1: The question asks for specific information regarding which related works contributed to the paper, which requires retrieving specific context from the MetaGPT paper rather than providing a general summary..
2026-07-14 10:33:39,533 - INFO - AFC is enabled with max remote calls: 10.


Selecting query engine 1: The question asks for specific information regarding which related works contributed to the paper, which requires retrieving specific context from the MetaGPT paper rather than providing a general summary..


2026-07-14 10:34:12,500 - INFO - HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemma-4-31b-it:generateContent "HTTP/1.1 200 OK"


The provided text does not specify which related work contributed the most to the paper. It mentions several works focusing on sociological phenomena, cooperation, competition, and LLM-based economies, and notes that the challenges of "assistant repeated instruction" or "infinite loop of message" identified by Talebirad & Nadiri (2023) and Li et al. (2023) motivated the focus on applying Standard Operating Procedures in software development to multi-agent frameworks.
